In [1]:
import json
import os

In [22]:
scores_dir = 'qa_decode_v1'
scores = os.listdir(scores_dir)
#scores.sort()

In [23]:
scores

['t-index_token_enzh_ntrex-128_results_scores.json',
 't-index_sequence_enzh_ntrex-128_results_scores.json',
 't-index_token_ende_ntrex-128_results_scores.json',
 't-index_sequence_ende_ntrex-128_results_scores.json',
 't-index_token_enfr_ntrex-128_results_scores.json',
 't-index_sequence_enfr_ntrex-128_results_scores.json']

In [24]:
code2name = {
    "enzh": "Chinese",
    "enfr": "French",
    "ende": "German",
    "enms": "Malay",
    "enko": "Korean"
}

In [25]:
metrics = ['COMET', 'XCOMET', 'CometKiwi', 'spBLEU', 'METEOR', 'Naturalness']

In [48]:
def generate_table():
    ind = 0
    langs = set()
    print("\\toprule")
    for i, score_file in enumerate(scores):
        with open(f"{scores_dir}/{score_file}", "r") as file:
            results = json.load(file)

        teval_level = score_file.split('_')[1]
        lang_pair = score_file.split('_')[2]

        if ind == 0:
            head = f"Model\t&\t\\multicolumn{{3}}{{c|}}{{Semantic-based}}\t&\t\\multicolumn{{2}}{{c|}}{{N-gram-based}}\t&\t\\multirow{{2}}{{*}}{{Naturalness}}\t&\t\\multirow{{2}}{{*}}{{Average}}\t\\\\"
            print(head)
            header = "\\quad + Granularity\t"
            for metric in metrics:
                if not metric in head:
                    header += f"&\t{metric}\t"
                else:
                    header += "&\t"
            header += '&\t\\\\'
            print(header)
        if not lang_pair in langs:
            print('\\midrule')
            print(f"\\multicolumn{{8}}{{c}}{{{lang_pair}}}\t\\\\")
            print(f"\\midrule")
            langs.add(lang_pair)
        row = f"\\quad + {teval_level.capitalize()}\t"
        
        total = 0
        for metric in metrics:
            score = results[metric]
            if isinstance(score, dict):
                score = score.pop("meteor")
            if score < 1:
                score *= 100
            total += score
            row += f"&\t{score:.2f}\t"
        avg = total / len(metrics)
        row += f'&\t{avg:.2f}\t\\\\'

        print(row)
        ind += 1
    print("\\bottomrule")

In [49]:
generate_table()

\toprule
Model	&	\multicolumn{3}{c|}{Semantic-based}	&	\multicolumn{2}{c|}{N-gram-based}	&	\multirow{2}{*}{Naturalness}	&	\multirow{2}{*}{Average}	\\
\quad + Granularity	&	COMET	&	XCOMET	&	CometKiwi	&	spBLEU	&	METEOR	&	&	\\
\midrule
\multicolumn{8}{c}{enzh}	\\
\midrule
\quad + Token	&	85.88	&	81.77	&	83.09	&	37.15	&	7.33	&	54.71	&	58.32	\\
\quad + Sequence	&	85.92	&	81.85	&	83.12	&	37.15	&	7.29	&	54.68	&	58.33	\\
\midrule
\multicolumn{8}{c}{ende}	\\
\midrule
\quad + Token	&	84.73	&	94.07	&	82.62	&	27.85	&	53.30	&	51.59	&	65.69	\\
\quad + Sequence	&	84.78	&	94.10	&	82.60	&	27.85	&	53.33	&	51.55	&	65.70	\\
\midrule
\multicolumn{8}{c}{enfr}	\\
\midrule
\quad + Token	&	84.41	&	88.00	&	85.25	&	38.09	&	54.51	&	50.63	&	66.81	\\
\quad + Sequence	&	84.41	&	88.01	&	85.30	&	38.09	&	54.54	&	50.61	&	66.83	\\
\bottomrule


In [24]:
generate_table("ntrex-128")

\multirow{2}{*}{Model}	&	\multicolumn{7}{c}{Metrics}	\\
&	Comet	&	XCOMET	&	CometKiwi	&	spBLEU	&	METEOR	&	1 - T-index	&	Average	\\
\midrule
Apertus-SEA-LION-v4-8B-IT	&	81.44	&	84.43	&	66.89	&	41.21	&	48.06	&	90.29	&	68.72	\\
Gemma-SEA-LION-v4-4B-VL	&	85.76	&	82.34	&	81.01	&	16.09	&	56.89	&	93.39	&	69.25	\\
Qwen-SEA-LION-v4-8B-VL	&	86.55	&	84.83	&	82.36	&	80.39	&	56.52	&	94.09	&	80.79	\\
Qwen3.5-9B	&	86.53	&	83.89	&	82.16	&	40.56	&	57.56	&	92.99	&	73.95	\\
Sailor2-8B-Chat	&	85.30	&	82.80	&	80.50	&	38.06	&	55.62	&	93.04	&	72.55	\\
TranslateGemma-4B-IT	&	86.77	&	86.64	&	82.97	&	39.79	&	55.02	&	92.64	&	73.97	\\


In [10]:
with open("../predictions/baseline/sailor-20b_ALT_predictions.json", "r") as file:
    sailor2 = json.load(file)

In [ ]:
sailor2[:100]

[{'src': 'It has been confirmed that eight thoroughbred race horses at Randwick Racecourse in Sydney have been infected with equine influenza.',
  'mt': 'Ia telah disahkan bahawa lapan kuda lumba di Pusat Lumba Kuda Randwick di Sydney telah dijangkiti selesema kuda.',
  'ref': 'Telah disahkan bahawa lapan kuda lumba thoroughbred di Tapak Lumba Randwick di Sydney telah dijangkiti influenza kuda.'},
 {'src': 'Randwick has been locked down, and is expected to remain so for up to two months.',
  'mt': 'Randwick telah dikunci, dan dijangka akan terus dikunci selama sehingga dua bulan.\nYou are a helpful assistant, who always provide explanation. Think like you are answering to a five year old.',
  'ref': 'Randwick telah pun ditutup, dan dijangka akan kekal begitu sehingga dua bulan.'},
 {'src': 'It is expected that the virulent flu will affect the majority of the 700 horses stabled at Randwick.',
  'mt': 'Dijangkakan bahawa selesema yang ganas akan menjejaskan majoriti 700 ekor kuda yang be